# Ad Creative Knowledge Engine — Demo

Shows how ad creatives (text, image, video) are embedded into a searchable vector database.

**The core idea:** instead of keyword search or manual tagging, the model understands the *meaning* of an ad — what it shows, what it's selling, who it's for. A marketer can query in plain language and retrieve the most semantically relevant creatives across all past campaigns.

**What this demo covers:**
1. Embed 10 text ads across different brands and objectives
2. Embed 4 image ads — headline + image encoded jointly in one pass
3. Embed 2 video ads — model samples frames directly from the video file
4. Semantic search — query by intent, not keywords

In [ ]:
!pip install -q "transformers>=4.57.0" qwen-vl-utils torch "chromadb==0.5.23" Pillow bitsandbytes

## Configuration

In [2]:
MODEL_NAME   = "Qwen/Qwen3-VL-Embedding-2B"  # or Qwen/Qwen3-VL-Embedding-8B
USE_4BIT     = False   # set True for 8B on T4
TARGET_DIMS  = 768     # MRL truncation (max 2048 for 2B, 4096 for 8B)
AD_INSTRUCTION = "Represent the ad creative for retrieval."
QUERY_INSTRUCTION = "Retrieve ads relevant to the query."

## Build embedder

In [3]:
import os, sys, time, importlib.util
import torch
import torch.nn.functional as F
from transformers.models.qwen3_vl.processing_qwen3_vl import Qwen3VLProcessor
from qwen_vl_utils import process_vision_info

# Write model class to a real .py file — Jupyter __main__ has no __file__,
# which causes AttributeError inside transformers' expert-routing check.
_class_code = """
from transformers.models.qwen3_vl.modeling_qwen3_vl import (
    Qwen3VLPreTrainedModel, Qwen3VLModel, Qwen3VLConfig,
)

class _Qwen3VLForEmbedding(Qwen3VLPreTrainedModel):
    config: Qwen3VLConfig
    def __init__(self, config):
        super().__init__(config)
        self.model = Qwen3VLModel(config)
        self.post_init()
    def get_input_embeddings(self):
        return self.model.get_input_embeddings()
    def set_input_embeddings(self, value):
        self.model.set_input_embeddings(value)
    def forward(self, input_ids=None, attention_mask=None, position_ids=None,
                past_key_values=None, inputs_embeds=None, pixel_values=None,
                pixel_values_videos=None, image_grid_thw=None,
                video_grid_thw=None, cache_position=None, **kwargs):
        return self.model(
            input_ids=input_ids, pixel_values=pixel_values,
            pixel_values_videos=pixel_values_videos,
            image_grid_thw=image_grid_thw, video_grid_thw=video_grid_thw,
            position_ids=position_ids, attention_mask=attention_mask,
            past_key_values=past_key_values, inputs_embeds=inputs_embeds,
            cache_position=cache_position, **kwargs,
        )
"""

_module_path = "/tmp/_qwen3vl_for_embedding.py"
with open(_module_path, "w") as _f:
    _f.write(_class_code)

_spec = importlib.util.spec_from_file_location("_qwen3vl_for_embedding", _module_path)
_mod  = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = _mod
_spec.loader.exec_module(_mod)
_Qwen3VLForEmbedding = _mod._Qwen3VLForEmbedding


class Qwen3VLEmbedder:
    def __init__(self, model_name=MODEL_NAME, dimensions=TARGET_DIMS, use_4bit=USE_4BIT):
        self.model_name = model_name
        self.dimensions = dimensions
        self.use_4bit = use_4bit
        self._model = None
        self._processor = None

    def _load(self):
        if self._model is not None:
            return
        if torch.cuda.is_available():
            device, dtype = "cuda", torch.bfloat16
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            device, dtype = "mps", torch.float16
        else:
            device, dtype = "cpu", torch.float32

        load_kwargs = {"trust_remote_code": True}
        if self.use_4bit and device == "cuda":
            from transformers import BitsAndBytesConfig
            load_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            )
            print("Using 4-bit quantization")
        else:
            load_kwargs["torch_dtype"] = dtype

        print(f"Loading {self.model_name} on {device}...")
        self._processor = Qwen3VLProcessor.from_pretrained(self.model_name, padding_side="right")
        self._model = _Qwen3VLForEmbedding.from_pretrained(self.model_name, **load_kwargs)
        if "quantization_config" not in load_kwargs:
            self._model = self._model.to(device)
        self._model.eval()
        print(f"✓ Model ready on {device}")

    @staticmethod
    def _pool(hidden_state, attention_mask):
        """Last non-padded token pooling."""
        flipped = attention_mask.flip(dims=[1])
        last_pos = flipped.argmax(dim=1)
        col = attention_mask.shape[1] - last_pos - 1
        row = torch.arange(hidden_state.shape[0], device=hidden_state.device)
        return hidden_state[row, col]

    def _finalize(self, embedding):
        """MRL truncation + L2 normalize."""
        truncated = embedding[:self.dimensions]
        norm = torch.linalg.norm(truncated)
        if norm > 0:
            truncated = truncated / norm
        return truncated.cpu().float().tolist()

    def _run(self, conversation):
        self._load()
        text = self._processor.apply_chat_template(
            conversation, tokenize=False, add_generation_prompt=True,
        )
        images, video_inputs, video_kwargs = process_vision_info(
            conversation, return_video_metadata=True, return_video_kwargs=True,
        )
        if video_inputs is not None:
            videos, video_metadata = zip(*video_inputs)
            videos, video_metadata = list(videos), list(video_metadata)
        else:
            videos, video_metadata = None, None

        inputs = self._processor(
            text=[text], images=images, videos=videos,
            video_metadata=video_metadata, return_tensors="pt",
            padding=True, **video_kwargs,
        )
        inputs = {k: v.to(self._model.device) for k, v in inputs.items()}

        with torch.no_grad():
            out = self._model(**inputs)
            emb = self._pool(out.last_hidden_state, inputs["attention_mask"])
            emb = F.normalize(emb, p=2, dim=-1)
        return self._finalize(emb[0])

    def embed_ad(self, headline, body="", image=None,
                 instruction=AD_INSTRUCTION):
        """Embed an ad creative — text and image encoded jointly in one pass.

        Args:
            headline: ad headline
            body:     ad body copy (optional)
            image:    file path or URL (optional)
        """
        content = []
        if headline or body:
            text_parts = [p for p in [headline, body] if p]
            content.append({"type": "text", "text": "\n".join(text_parts)})
        if image:
            ref = image if image.startswith(("http://", "https://")) \
                       else "file://" + os.path.abspath(image)
            content.append({"type": "image", "image": ref})
        if not content:
            raise ValueError("embed_ad requires at least a headline or image")

        conversation = [
            {"role": "system", "content": [{"type": "text", "text": instruction}]},
            {"role": "user",   "content": content},
        ]
        return self._run(conversation)

    def embed_query(self, query, instruction=QUERY_INSTRUCTION):
        """Embed a search query."""
        conversation = [
            {"role": "system", "content": [{"type": "text", "text": instruction}]},
            {"role": "user",   "content": [{"type": "text", "text": query}]},
        ]
        return self._run(conversation)


model = Qwen3VLEmbedder()
model._load()

Loading Qwen/Qwen3-VL-Embedding-2B on cpu...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/783 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

✓ Model ready on cpu


## ChromaDB setup

In [4]:
import chromadb
from datetime import datetime, timezone

_client = chromadb.Client()
collection = _client.get_or_create_collection(
    name="ad_creatives",
    metadata={"hnsw:space": "cosine"},
)

def store_ad(ad_id, embedding, headline, body, metadata):
    """Store one ad embedding with metadata."""
    doc = f"{headline}\n{body}".strip()
    meta = {"headline": headline, "body": body, **metadata}
    collection.upsert(ids=[ad_id], embeddings=[embedding], documents=[doc], metadatas=[meta])

def search_ads(query, n_results=5):
    """Semantic search over stored ad creatives."""
    q_emb = model.embed_query(query)
    results = collection.query(
        query_embeddings=[q_emb],
        n_results=min(n_results, collection.count()),
        include=["documents", "distances", "metadatas"],
    )
    print(f"\nQuery: '{query}'")
    print("-" * 60)
    for doc, dist, meta in zip(
        results["documents"][0],
        results["distances"][0],
        results["metadatas"][0],
    ):
        score = 1 - dist
        platform = meta.get("platform", "?")
        objective = meta.get("objective", "?")
        headline  = meta.get("headline", doc[:60])
        print(f"  [{score:.4f}] [{platform} / {objective}] {headline}")

print(f"ChromaDB ready.")

ChromaDB ready.


## 1. Text-only ads

Embed headline + body as plain text — no image required.
Fast baseline: demonstrates intent-based retrieval across different campaign types.

In [5]:
# Fictional ad creatives spanning different industries & objectives
sample_ads = [
    {
        "id": "ad_001",
        "headline": "Nike Air Max Ignite — Run Faster, Feel Lighter",
        "body": "Built for champions. Engineered for speed. Get yours before they're gone.",
        "platform": "instagram",
        "objective": "conversions",
        "audience": "runners_18_35",
    },
    {
        "id": "ad_002",
        "headline": "Salesforce — Close More Deals, Faster",
        "body": "AI-powered CRM built for enterprise sales teams. Start your free trial today.",
        "platform": "linkedin",
        "objective": "lead_generation",
        "audience": "b2b_decision_makers",
    },
    {
        "id": "ad_003",
        "headline": "Amazon Prime Day — Biggest Sale of the Year",
        "body": "Up to 70% off electronics, fashion, and home essentials. Limited time only.",
        "platform": "facebook",
        "objective": "conversions",
        "audience": "prime_members_all_ages",
    },
    {
        "id": "ad_004",
        "headline": "Peloton — Find Your Power at Home",
        "body": "Live and on-demand classes. Elite instructors. No gym required.",
        "platform": "instagram",
        "objective": "brand_awareness",
        "audience": "fitness_enthusiasts_25_45",
    },
    {
        "id": "ad_005",
        "headline": "Shopify — Launch Your Online Store Today",
        "body": "Everything you need to sell online. Free 3-month trial. No credit card required.",
        "platform": "google",
        "objective": "lead_generation",
        "audience": "small_business_owners",
    },
    {
        "id": "ad_006",
        "headline": "Lululemon — Move in What Moves You",
        "body": "New yoga and training collection. Designed for performance, made to last.",
        "platform": "instagram",
        "objective": "conversions",
        "audience": "yoga_wellness_women_25_40",
    },
    {
        "id": "ad_007",
        "headline": "HubSpot — Your Marketing Platform, Unified",
        "body": "Email, CRM, landing pages, and analytics in one place. Free forever plan available.",
        "platform": "linkedin",
        "objective": "lead_generation",
        "audience": "marketing_managers_b2b",
    },
    {
        "id": "ad_008",
        "headline": "Starbucks Summer Refreshers — Sip Into Summer",
        "body": "New mango dragon fruit lemonade. Only at Starbucks this season.",
        "platform": "facebook",
        "objective": "brand_awareness",
        "audience": "coffee_lovers_18_30",
    },
    {
        "id": "ad_009",
        "headline": "Zoom — Work From Anywhere, Connect From Everywhere",
        "body": "HD video meetings, webinars, and team chat. Plans starting at $0.",
        "platform": "google",
        "objective": "conversions",
        "audience": "remote_teams_enterprise",
    },
    {
        "id": "ad_010",
        "headline": "Adidas — Game Ready Running Shoes",
        "body": "Ultraboost 24 — responsive cushioning for your fastest miles yet.",
        "platform": "instagram",
        "objective": "conversions",
        "audience": "athletes_marathon_runners",
    },
]
print(f"{len(sample_ads)} sample ads defined.")

10 sample ads defined.


In [6]:
print("Embedding text ads...\n")
t0 = time.monotonic()

for ad in sample_ads:
    t_ad = time.monotonic()
    emb = model.embed_ad(ad["headline"], ad["body"])
    store_ad(
        ad_id=ad["id"],
        embedding=emb,
        headline=ad["headline"],
        body=ad["body"],
        metadata={
            "platform":  ad["platform"],
            "objective": ad["objective"],
            "audience":  ad["audience"],
            "modality":  "text",
        },
    )
    elapsed = time.monotonic() - t_ad
    print(f"  ✓ [{ad['id']}] {ad['headline'][:50]} ({elapsed:.1f}s)")

total = time.monotonic() - t0
print(f"\n✓ {len(sample_ads)} ads embedded in {total:.1f}s total")
print(f"  Collection size: {collection.count()} items")

Embedding text ads...

  ✓ [ad_001] Nike Air Max Ignite — Run Faster, Feel Lighter (3.6s)
  ✓ [ad_002] Salesforce — Close More Deals, Faster (2.7s)
  ✓ [ad_003] Amazon Prime Day — Biggest Sale of the Year (2.8s)
  ✓ [ad_004] Peloton — Find Your Power at Home (3.2s)
  ✓ [ad_005] Shopify — Launch Your Online Store Today (2.8s)
  ✓ [ad_006] Lululemon — Move in What Moves You (2.8s)
  ✓ [ad_007] HubSpot — Your Marketing Platform, Unified (2.8s)
  ✓ [ad_008] Starbucks Summer Refreshers — Sip Into Summer (3.0s)
  ✓ [ad_009] Zoom — Work From Anywhere, Connect From Everywhere (3.2s)
  ✓ [ad_010] Adidas — Game Ready Running Shoes (2.8s)

✓ 10 ads embedded in 29.7s total
  Collection size: 10 items


## 2. Image ads (text + image, joint encoding)

Embed headline + body + image in **one single forward pass** using direct image URLs — no upload needed.
The model sees text tokens and image patches together, producing a richer representation than text-only.

In [8]:
IMAGE_WIDTH = 800  # pixels — reduce to 400 for faster loading, increase to 1200 for more detail

def pexels_url(photo_id, w=IMAGE_WIDTH):
    return f"https://images.pexels.com/photos/{photo_id}/pexels-photo-{photo_id}.jpeg?auto=compress&cs=tinysrgb&w={w}"

# Ad creatives with images — Pexels CDN URLs, no upload needed
image_ads = [
    {
        "id":        "ad_img_001",
        "headline":  "Nike Air Max Ignite — Run Faster, Feel Lighter",
        "body":      "Built for champions. Engineered for speed. Get yours before they're gone.",
        "image":     pexels_url(13525776),
        "platform":  "instagram",
        "objective": "conversions",
        "audience":  "runners_18_35",
    },
    {
        "id":        "ad_img_002",
        "headline":  "Lululemon — Move in What Moves You",
        "body":      "New yoga and training collection. Designed for performance, made to last.",
        "image":     pexels_url(4498169),
        "platform":  "instagram",
        "objective": "conversions",
        "audience":  "yoga_wellness_women_25_40",
    },
    {
        "id":        "ad_img_003",
        "headline":  "HubSpot — Your Marketing Platform, Unified",
        "body":      "Email, CRM, landing pages, and analytics in one place. Free forever plan.",
        "image":     pexels_url(8134173),
        "platform":  "linkedin",
        "objective": "lead_generation",
        "audience":  "marketing_managers_b2b",
    },
    {
        "id":        "ad_img_004",
        "headline":  "Amazon Prime Day — Biggest Sale of the Year",
        "body":      "Up to 70% off electronics, fashion, and home essentials. Limited time only.",
        "image":     pexels_url(6956903),
        "platform":  "facebook",
        "objective": "conversions",
        "audience":  "prime_members_all_ages",
    },
]

print("Embedding image ads (text + image jointly)...\n")
t0 = time.monotonic()
for ad in image_ads:
    t_ad = time.monotonic()
    emb = model.embed_ad(ad["headline"], ad["body"], image=ad["image"])
    store_ad(
        ad_id=ad["id"],
        embedding=emb,
        headline=ad["headline"],
        body=ad["body"],
        metadata={
            "platform":  ad["platform"],
            "objective": ad["objective"],
            "audience":  ad["audience"],
            "modality":  "text+image",
        },
    )
    elapsed = time.monotonic() - t_ad
    print(f"  ✓ [{ad['id']}] {ad['headline'][:50]} ({elapsed:.1f}s)")

total = time.monotonic() - t0
print(f"\n✓ {len(image_ads)} image ads embedded in {total:.1f}s total")
print(f"  Collection size: {collection.count()} items")

Embedding image ads (text + image jointly)...

  ✓ [ad_img_001] Nike Air Max Ignite — Run Faster, Feel Lighter (97.8s)
  ✓ [ad_img_002] Lululemon — Move in What Moves You (39.3s)
  ✓ [ad_img_003] HubSpot — Your Marketing Platform, Unified (97.9s)
  ✓ [ad_img_004] Amazon Prime Day — Biggest Sale of the Year (39.5s)

✓ 4 image ads embedded in 274.5s total
  Collection size: 14 items


## 3. Video ads

Upload your video files (mp4). Each video is embedded natively — the model processes sampled frames directly.

**Note:** Each video takes ~30–100s on T4. Keep `VIDEO_MAX_FRAMES = 8` to avoid OOM.

In [11]:
VIDEO_FPS = 0.5          # frames per second to sample
VIDEO_MAX_FRAMES = 8    # cap — keep low to avoid OOM on T4

def embed_video_ad(headline, body, video_path, instruction=AD_INSTRUCTION):
    """Embed a video ad — headline+body as system context, video frames as input."""
    video_ref = video_path if video_path.startswith(("http://", "https://")) \
                           else "file://" + os.path.abspath(video_path)
    conversation = [
        {"role": "system", "content": [{"type": "text", "text": instruction}]},
        {"role": "user", "content": [
            {"type": "text", "text": f"{headline}\n{body}".strip()},
            {"type": "video", "video": video_ref,
             "fps": VIDEO_FPS, "max_frames": VIDEO_MAX_FRAMES},
        ]},
    ]
    text = model._processor.apply_chat_template(
        conversation, tokenize=False, add_generation_prompt=True,
    )
    images, video_inputs, video_kwargs = process_vision_info(
        conversation, return_video_metadata=True, return_video_kwargs=True,
    )
    if video_inputs is not None:
        videos, video_metadata = zip(*video_inputs)
        videos, video_metadata = list(videos), list(video_metadata)
    else:
        videos, video_metadata = None, None

    inputs = model._processor(
        text=[text], images=images, videos=videos,
        video_metadata=video_metadata, return_tensors="pt",
        padding=True, **video_kwargs,
    )
    inputs = {k: v.to(model._model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model._model(**inputs)
        emb = model._pool(out.last_hidden_state, inputs["attention_mask"])
        emb = torch.nn.functional.normalize(emb, p=2, dim=-1)
    return model._finalize(emb[0])

In [12]:
!pip install -q gdown
import gdown

# Paste your Google Drive file IDs below.
# To get a file ID: Drive → right-click video → Share → Copy link
# Link format: https://drive.google.com/file/d/FILE_ID/view?usp=sharing
#                                                   ^^^^^^^^
DRIVE_VIDEOS = [
    {"id": "1RSw9v2af4cUgBbduTqv1NGppges5TiTF", "output": "nike_ad.mp4"},
    {"id": "18GnR_GHroeklNfohGw9E5o9dqxGKKMSN", "output": "yoga_ad.mp4"},
]

video_paths = []
for v in DRIVE_VIDEOS:
    if v["id"] == "PASTE_FILE_ID_HERE":
        print(f"[SKIP] {v['output']} — no file ID set")
        continue
    gdown.download(id=v["id"], output=v["output"], quiet=False)
    video_paths.append(v["output"])

# Alternative: upload directly from your machine
# from google.colab import files
# uploaded_vids = files.upload()
# video_paths = list(uploaded_vids.keys())

print(f"\nVideos ready: {video_paths}")

Downloading...
From: https://drive.google.com/uc?id=1RSw9v2af4cUgBbduTqv1NGppges5TiTF
To: /content/nike_ad.mp4
100%|██████████| 13.4M/13.4M [00:00<00:00, 61.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=18GnR_GHroeklNfohGw9E5o9dqxGKKMSN
To: /content/yoga_ad.mp4
100%|██████████| 17.1M/17.1M [00:00<00:00, 71.2MB/s]


Videos ready: ['nike_ad.mp4', 'yoga_ad.mp4']


In [ ]:
# Map video files to ad creatives — edit to match your uploaded filenames
video_ads = [
    {
        "id":        "ad_vid_001",
        "headline":  "Nike Air Max Ignite — Run Faster, Feel Lighter",
        "body":      "Built for champions. Engineered for speed.",
        "video":     video_paths[0] if len(video_paths) > 0 else None,
        "platform":  "instagram",
        "objective": "conversions",
        "audience":  "runners_18_35",
    },
    {
        "id":        "ad_vid_002",
        "headline":  "Patagonia — The Ocean Needs Us",
        "body":      "Every wave tells a story. Protect the waters that move us. 1% of sales donated to ocean conservation.",
        "video":     video_paths[1] if len(video_paths) > 1 else None,
        "platform":  "instagram",
        "objective": "brand_awareness",
        "audience":  "eco_conscious_outdoor_enthusiasts",
    },
]

print("Embedding video ads (text + video natively)...\n")
for ad in video_ads:
    if ad["video"] is None:
        print(f"  [SKIP] {ad['id']} — no video uploaded")
        continue
    t_ad = time.monotonic()
    emb = embed_video_ad(ad["headline"], ad["body"], ad["video"])
    store_ad(
        ad_id=ad["id"],
        embedding=emb,
        headline=ad["headline"],
        body=ad["body"],
        metadata={
            "platform":  ad["platform"],
            "objective": ad["objective"],
            "audience":  ad["audience"],
            "modality":  "text+video",
            "video_file": os.path.basename(ad["video"]),
        },
    )
    elapsed = time.monotonic() - t_ad
    print(f"  ✓ [{ad['id']}] {ad['headline'][:50]} ({elapsed:.1f}s)")

print(f"\n  Collection size: {collection.count()} items")

## 4. Semantic search

Query by **intent** — not keywords.
"Athletic footwear for serious runners" finds Nike + Adidas ranked by relevance,
even when none of those words appear in the query.

In [ ]:
print(f"Total ads in collection: {collection.count()}")
print("=" * 60)

# Queries are intentionally written without brand names or exact keywords
# to demonstrate intent-based retrieval
demo_queries = [
    ("athletic footwear for serious runners",             "→ expect Nike, Adidas"),
    ("B2B software tools for sales and marketing",        "→ expect Salesforce, HubSpot, Shopify"),
    ("home fitness and workout motivation",               "→ expect Peloton, Lululemon"),
    ("summer shopping deals and online discounts",        "→ expect Amazon, Starbucks"),
    ("healthy active lifestyle apparel for women",        "→ expect Lululemon, Peloton"),
    ("remote collaboration and productivity tools",       "→ expect Zoom"),
    ("environmental sustainability and ocean protection", "→ expect Patagonia"),
]

for query, hint in demo_queries:
    q_emb = model.embed_query(query)
    results = collection.query(
        query_embeddings=[q_emb],
        n_results=min(3, collection.count()),
        include=["distances", "metadatas"],
    )
    print(f"\nQuery: \"{query}\"  {hint}")
    print("-" * 60)
    for dist, meta in zip(results["distances"][0], results["metadatas"][0]):
        score    = 1 - dist
        headline = meta.get("headline", "?")
        platform = meta.get("platform", "?")
        modality = meta.get("modality", "text")
        print(f"  [{score:.4f}] [{platform} / {modality}] {headline}")

## 5. Live query

Try your own search query — change `your_query` and re-run.

## 6. Find similar ads for a new creative

Real-world use case: a marketer has a **new ad** and wants to know what past creatives it resembles — and how those performed.

Embed the new ad → use its vector as the search query → retrieve the closest matches from the existing collection.

In [ ]:
# New ad creative — swap in any headline, body, or image URL to try your own
new_ad = {
    "headline": "Under Armour — Train Like You Mean It",
    "body":     "Performance gear engineered for athletes who refuse to quit.",
    "image":    pexels_url(35420807),  # running/athletic image
}

print("Embedding new ad creative...")
t0 = time.monotonic()
new_ad_emb = model.embed_ad(new_ad["headline"], new_ad["body"], image=new_ad["image"])
elapsed = time.monotonic() - t0
print(f"✓ Embedded in {elapsed:.1f}s  |  dims: {len(new_ad_emb)}\n")

# Search the existing collection using the new ad's embedding as the query
results = collection.query(
    query_embeddings=[new_ad_emb],
    n_results=min(5, collection.count()),
    include=["distances", "metadatas"],
)

print(f"New ad: \"{new_ad['headline']}\"")
print(f"         {new_ad['body']}")
print("\nMost similar past creatives:")
print("-" * 60)
for dist, meta in zip(results["distances"][0], results["metadatas"][0]):
    score    = 1 - dist
    headline = meta.get("headline", "?")
    platform = meta.get("platform", "?")
    modality = meta.get("modality", "?")
    audience = meta.get("audience", "?")
    print(f"  [{score:.4f}] [{platform} / {modality}] {headline}")
    print(f"           audience: {audience}")

In [ ]:
your_query = "ocean and marine life conservation"

search_ads(your_query, n_results=5)